In [8]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ('LANGCHAIN_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = getpass.getpass("Enter LangSmith API Key: ")
    
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'rag-lab-query-translation'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
# Loading the Vectorstore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma   
from langchain_google_genai import ChatGoogleGenerativeAI

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini")
)
print("Chunks in store:", vectorstore._collection.count())

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

Chunks in store: 979


In [14]:
# Multi-query, hand-rolled with LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.documents import Document
from typing import List
from langchain_core.output_parsers import StrOutputParser

class LineListOutputParser(BaseOutputParser[List[str]]):
    """Splits an LLM's newline-separated output into a clean list of query strings."""

    def parse(self, text: str) -> List[str]:
        return [line.strip() for line in text.strip().split("\n") if line.strip()]


MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template(
    "You are an AI assistant. Generate 4 different rephrasings of the "
    "user question below, to help retrieve relevant documents from a "
    "vector database. Provide these alternative questions separated "
    "by newlines, with no numbering or extra commentary.\n\n"
    "Original question: {question}"
)

In [15]:
class MultiQueryStrategy:
    """Generates multiple rephrasings of a query, retrieves for each,
    and returns the deduplicated union of retrieved documents."""

    def __init__(self, vectorstore, llm, k: int = 4):
        self.vectorstore = vectorstore
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        self.llm = llm
        self.query_gen_chain = (
            MULTI_QUERY_PROMPT | llm | LineListOutputParser()
        )
        self.answer_prompt = ChatPromptTemplate.from_template(
            "Answer the question based only on the following context:\n"
            "{context}\n\nQuestion: {question}"
        )

    def generate_queries(self, query: str) -> List[str]:
        queries = self.query_gen_chain.invoke({"question": query})
        return [query] + queries   # include the original alongside rephrasings

    @staticmethod
    def _unique_union(doc_lists: List[List[Document]]) -> List[Document]:
        seen = set()
        unique_docs = []
        for docs in doc_lists:
            for doc in docs:
                key = doc.page_content
                if key not in seen:
                    seen.add(key)
                    unique_docs.append(doc)
        return unique_docs

    def retrieve(self, query: str) -> List[Document]:
        all_queries = self.generate_queries(query)
        doc_lists = [self.retriever.invoke(q) for q in all_queries]
        return self._unique_union(doc_lists)

    def run(self, query: str) -> str:
        docs = self.retrieve(query)
        context = "\n\n".join(d.page_content for d in docs)
        chain = self.answer_prompt | self.llm | StrOutputParser()
        return chain.invoke({"context": context, "question": query})

In [16]:
from rag_lab.strategies.multi_query import MultiQueryStrategy   

test_query = "Why do PINNs struggle with irregular geometry?"
strategy = MultiQueryStrategy(vectorstore, llm=llm)

print("Generated queries:")
for q in strategy.generate_queries(test_query):
    print(" -", q)

print("\nAnswer:")
print(strategy.run(test_query))

Generated queries:


/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


 - Why do PINNs struggle with irregular geometry?
 - What causes Physics-Informed Neural Networks to perform poorly on complex or non-Euclidean geometries?
 - What are the main difficulties PINNs face when dealing with irregular domain boundaries?
 - Why do physics-informed neural networks have trouble solving PDEs on non-standard or complex spatial domains?
 - What limits the effectiveness and accuracy of PINNs when applied to irregular and arbitrary shapes?

Answer:


/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Clie

Based on the provided context, standard PINNs struggle with irregular geometries and unstructured meshes for the following reasons:

* **Automatic Differentiation (AD) Limitations:** The Automatic Differentiation algorithm—typically used for uniform and structured meshes—struggles to efficiently propagate derivatives through complex and irregular connections found in unstructured meshes.
* **Complex Spatial Gradients:** Computing spatial gradients on unstructured meshes is more complex due to the irregular neighborhoods of their triangular cells.
* **Generalization Issues:** PINNs lack the ability to generalize to out-of-sample scenarios, such as varying computational domains, spatial resolutions, and time scales.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
